In [ ]:
!pip install apache-beam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
spark =  SparkSession.builder.appName('test').getOrCreate()

In [ ]:
df =  spark.read.csv('/content/sample - Sheet1 (1).csv', header=True)

In [ ]:
df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
#testing

import apache_beam as beam

data_captured = []

def process_row(row):
    """Splits CSV row and concatenates first and last name."""
    fields = row.split(",")
    if len(fields) < 6:
        return None

    full_name = f"{fields[1]} {fields[2]}"
    return (fields[0], fields[1], fields[2], full_name, fields[3], fields[4], fields[5], fields[6], fields[7])

def capture_data(data_list):
    global data_captured
    data_captured = data_list

with beam.Pipeline() as pipeline:
    transformed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv')
        | "Transform Data" >> beam.Map(process_row)
        #| "Filter None" >> beam.Filter(lambda x: x is not None)
        | "Collect to List" >> beam.combiners.ToList()
        | "captured the data" >> beam.Map(capture_data)
    )

    transformed_data | "Print Results" >> beam.Map(print)


None


In [ ]:
data_captured

[('id',
  'first_name',
  'last_name',
  'first_name last_name',
  'age',
  'salary',
  'department',
  'erdate',
  'udate'),
 ('1',
  'John',
  'Doe',
  'John Doe',
  '30',
  '60000',
  'IT',
  '2025-03-07',
  '2025-03-07'),
 ('2', 'Jane', 'Smith', 'Jane Smith', '25', '55000', 'HR', '2025-03-08', ''),
 ('3',
  'Michael',
  'Johnson',
  'Michael Johnson',
  '35',
  '75000',
  'Finance',
  '2025-03-09',
  '2025-03-09'),
 ('4',
  'Emily',
  'Davis',
  'Emily Davis',
  '28',
  '58000',
  'Marketing',
  '2025-03-10',
  ''),
 ('5',
  'Daniel',
  'Martinez',
  'Daniel Martinez',
  '40',
  '90000',
  'IT',
  '2025-03-11',
  '2025-03-11'),
 ('6',
  'Sophia',
  'Lopez',
  'Sophia Lopez',
  '27',
  '62000',
  'HR',
  '2025-03-12',
  ''),
 ('7',
  'James',
  'Gonzalez',
  'James Gonzalez',
  '45',
  '110000',
  'Finance',
  '2025-03-13',
  '2025-03-13'),
 ('8',
  'Olivia',
  'Wilson',
  'Olivia Wilson',
  '32',
  '70000',
  'Marketing',
  '2025-03-14',
  ''),
 ('9',
  'William',
  'Anderson',
  '

In [ ]:
column = data_captured[0]

In [ ]:
print(header)

full_name


In [ ]:
data =  data_captured[1:]

In [ ]:
data

[('1',
  'John',
  'Doe',
  'John Doe',
  '30',
  '60000',
  'IT',
  '2025-03-07',
  '2025-03-07'),
 ('2', 'Jane', 'Smith', 'Jane Smith', '25', '55000', 'HR', '2025-03-08', ''),
 ('3',
  'Michael',
  'Johnson',
  'Michael Johnson',
  '35',
  '75000',
  'Finance',
  '2025-03-09',
  '2025-03-09'),
 ('4',
  'Emily',
  'Davis',
  'Emily Davis',
  '28',
  '58000',
  'Marketing',
  '2025-03-10',
  ''),
 ('5',
  'Daniel',
  'Martinez',
  'Daniel Martinez',
  '40',
  '90000',
  'IT',
  '2025-03-11',
  '2025-03-11'),
 ('6',
  'Sophia',
  'Lopez',
  'Sophia Lopez',
  '27',
  '62000',
  'HR',
  '2025-03-12',
  ''),
 ('7',
  'James',
  'Gonzalez',
  'James Gonzalez',
  '45',
  '110000',
  'Finance',
  '2025-03-13',
  '2025-03-13'),
 ('8',
  'Olivia',
  'Wilson',
  'Olivia Wilson',
  '32',
  '70000',
  'Marketing',
  '2025-03-14',
  ''),
 ('9',
  'William',
  'Anderson',
  'William Anderson',
  '29',
  '63000',
  'IT',
  '2025-03-15',
  '2025-03-15'),
 ('10',
  'Ava',
  'Thomas',
  'Ava Thomas',
  

In [ ]:
df =  spark.createDataFrame(data, column)

In [ ]:
df.show()

+---+----------+---------+--------------------+---+------+----------+----------+----------+
| id|first_name|last_name|first_name last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+--------------------+---+------+----------+----------+----------+
|  1|      John|      Doe|            John Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith|          Jane Smith| 25| 55000|        HR|2025-03-08|          |
|  3|   Michael|  Johnson|     Michael Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis|         Emily Davis| 28| 58000| Marketing|2025-03-10|          |
|  5|    Daniel| Martinez|     Daniel Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez|        Sophia Lopez| 27| 62000|        HR|2025-03-12|          |
|  7|     James| Gonzalez|      James Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson|       Olivia Wilson| 32| 70000| Marketing|2025-03-14|

In [ ]:
#testing Scenario 1

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime
from pyspark.sql import SparkSession
import pandas as pd

# Initialize Spark Session
spark = SparkSession.builder.appName("IncrementalLoad").getOrCreate()

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        erdate = datetime.strptime(record[6], "%Y-%m-%d")  # 'erdate' is at index 6
        if erdate > self.last_load_date:
            yield record

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        udate = datetime.strptime(record[7], "%Y-%m-%d") if record[7] and record[7] != "NULL" else None  # 'udate' is at index 7
        if udate and udate > self.last_load_date:
            yield record

# Define the last load date
LAST_LOAD_DATE = "2025-03-10"

# Beam Pipeline
pipeline_options = PipelineOptions()
with beam.Pipeline(options=pipeline_options) as pipeline:

    # Read CSV and parse lines into lists
    source_data = (
        pipeline
        | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv', skip_header_lines=1)
        | "Parse CSV to List" >> beam.Map(lambda line: line.split(','))
    )

    # Filter Newly Inserted and Updated Records
    latest_records = source_data | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE))
    updated_records = source_data | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE))

    # Convert to lists for PySpark DataFrame creation
    latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
    updated_list = updated_records | "Collect Updated Data" >> beam.combiners.ToList()

    # Store data for conversion to PySpark
    def store_latest(data):
        global latest_data
        latest_data = data

    def store_updated(data):
        global updated_data
        updated_data = data

    latest_list | "Store Latest Records" >> beam.Map(store_latest)
    updated_list | "Store Updated Records" >> beam.Map(store_updated)

# Define Schema
schema = ["id", "first_name", "last_name", "age", "salary", "department", "erdate", "udate"]

# Convert Lists to PySpark DataFrames
latest_df = spark.createDataFrame(pd.DataFrame(latest_data, columns=schema))
updated_df = spark.createDataFrame(pd.DataFrame(updated_data, columns=schema))

# Show DataFrames
print("Latest Inserted Records:")
latest_df.show()

print("\nUpdated Records:")
updated_df.show()


Latest Inserted Records:
+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|          |
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|          |
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|          |
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 14| Charlotte|    White| 33| 72000|        HR|2025-03-20|          |
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025

In [ ]:
#tested for incremental load 1 scenario2


import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime

class FilterIncremental(beam.DoFn):
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        erdate = datetime.strptime(record['erdate'], "%Y-%m-%d")
        udate = datetime.strptime(record['udate'], "%Y-%m-%d") if record['udate'] and record['udate'] != "NULL" else None

        # Include records if they are newly inserted or updated since last load
        if erdate > self.last_load_date or (udate and udate > self.last_load_date):
            yield record

# Define the last load date (change dynamically)
LAST_LOAD_DATE = "2025-03-24"

# Beam Pipeline
pipeline_options = PipelineOptions()
with beam.Pipeline(options=pipeline_options) as pipeline:

    # Read CSV and parse lines into dictionaries
    source_data = (
        pipeline
        | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv', skip_header_lines=1)
        | "Parse CSV to Dict" >> beam.Map(lambda line: dict(zip(
            ["id", "first_name", "last_name", "age", "salary", "department", "erdate", "udate"],
            line.split(',')
        )))
    )

    # Filter Incremental Data
    incremental_data = source_data | "Filter Incremental Records" >> beam.ParDo(FilterIncremental(LAST_LOAD_DATE))

    # Print or Write Output
    incremental_data | "Print Latest Data" >> beam.Map(print)  # Replace with actual sink


{'id': '19', 'first_name': 'Mason', 'last_name': 'Hall', 'age': '34', 'salary': '74000', 'department': 'Finance', 'erdate': '2025-03-25', 'udate': '2025-03-25'}
{'id': '20', 'first_name': 'Evelyn', 'last_name': 'Allen', 'age': '27', 'salary': '60000', 'department': 'Marketing', 'erdate': '2025-03-25', 'udate': ''}


In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime
import pandas as pd

class FilterLatest(beam.DoFn):
    """Filter records that are newly inserted (based on erdate)."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        erdate = datetime.strptime(record['erdate'], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record

class FilterUpdated(beam.DoFn):
    """Filter records that are updated (based on udate)."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        udate = datetime.strptime(record['udate'], "%Y-%m-%d") if record['udate'] and record['udate'] != "NULL" else None
        if udate and udate > self.last_load_date:
            yield record

# Define the last load date
LAST_LOAD_DATE = "2025-03-10"

# Beam Pipeline
pipeline_options = PipelineOptions()
with beam.Pipeline(options=pipeline_options) as pipeline:

    # Read CSV and parse lines into dictionaries
    source_data = (
        pipeline
        | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv', skip_header_lines=1)
        | "Parse CSV to Dict" >> beam.Map(lambda line: dict(zip(
            ["id", "first_name", "last_name", "age", "salary", "department", "erdate", "udate"],
            line.split(',')
        )))
    )

    # Separate Newly Inserted and Updated Records
    latest_records = source_data | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE))
    updated_records = source_data | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE))

    # Convert to lists for DataFrame creation
    latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
    updated_list = updated_records | "Collect Updated Data" >> beam.combiners.ToList()

    # Store into global variables (outside pipeline execution)
    def store_latest(data):
        global latest_df
        latest_df = pd.DataFrame(data)

    def store_updated(data):
        global updated_df
        updated_df = pd.DataFrame(data)

    latest_list | "Store Latest Records" >> beam.Map(store_latest)
    updated_list | "Store Updated Records" >> beam.Map(store_updated)



In [ ]:
# Print DataFrames (for testing)
print("Latest Inserted Records:")
print(latest_df)

print("\nUpdated Records:")
print(updated_df)

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime
import pandas as pd

class FilterLatest(beam.DoFn):
    """Filter records that are newly inserted (based on erdate)."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        erdate = datetime.strptime(record['erdate'], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record

class FilterUpdated(beam.DoFn):
    """Filter records that are updated (based on udate)."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        udate = datetime.strptime(record['udate'], "%Y-%m-%d") if record['udate'] and record['udate'] != "NULL" else None
        if udate and udate > self.last_load_date:
            yield record

# Define the last load date
LAST_LOAD_DATE = "2025-03-10"

# Beam Pipeline
pipeline_options = PipelineOptions()
with beam.Pipeline(options=pipeline_options) as pipeline:

    # Read CSV and parse lines into dictionaries
    source_data = (
        pipeline
        | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv', skip_header_lines=1)
        | "Parse CSV to Dict" >> beam.Map(lambda line: dict(zip(
            ["id", "first_name", "last_name", "age", "salary", "department", "erdate", "udate"],
            line.split(',')
        )))
    )

    ## Separate Newly Inserted and Updated Records
    latest_records = source_data | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE))
    updated_records = source_data | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE))

    # Convert to lists for DataFrame creation
    latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
    updated_list = updated_records | "Collect Updated Data" >> beam.combiners.ToList()

    ## Store into global variables (outside pipeline execution)
    #def store_latest(data):
    #    global latest_df
    #    latest_df = pd.DataFrame(data)
    #
    #def store_updated(data):
    #    global updated_df
    #    updated_df = pd.DataFrame(data)

    latest_list | "Store Latest Records" >> beam.Map(store_latest)
    updated_list | "Store Updated Records" >> beam.Map(store_updated)
    updated_list | "print the thinngs" >> beam.Map(print)


[{'id': '5', 'first_name': 'Daniel', 'last_name': 'Martinez', 'age': '40', 'salary': '90000', 'department': 'IT', 'erdate': '2025-03-11', 'udate': '2025-03-11'}, {'id': '7', 'first_name': 'James', 'last_name': 'Gonzalez', 'age': '45', 'salary': '110000', 'department': 'Finance', 'erdate': '2025-03-13', 'udate': '2025-03-13'}, {'id': '9', 'first_name': 'William', 'last_name': 'Anderson', 'age': '29', 'salary': '63000', 'department': 'IT', 'erdate': '2025-03-15', 'udate': '2025-03-15'}, {'id': '10', 'first_name': 'Ava', 'last_name': 'Thomas', 'age': '26', 'salary': '59000', 'department': 'HR', 'erdate': '2025-03-16', 'udate': '2025-03-16'}, {'id': '12', 'first_name': 'Mia', 'last_name': 'Moore', 'age': '24', 'salary': '54000', 'department': 'Marketing', 'erdate': '2025-03-18', 'udate': '2025-03-18'}, {'id': '13', 'first_name': 'Ethan', 'last_name': 'Jackson', 'age': '31', 'salary': '67000', 'department': 'IT', 'erdate': '2025-03-19', 'udate': '2025-03-19'}, {'id': '15', 'first_name': 'Be